# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv 
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from pathlib import Path

In [ ]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

from pathlib import Path
_env_path = next(
    (p for p in [Path('.env'), Path('project/starter/.env')] if p.exists()),
    Path('.env')
)
load_dotenv(dotenv_path=_env_path, override=True)


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

In [ ]:
# Instantiate ChromaDB Client
chroma_client = chromadb.PersistentClient(path="chromadb")

In [ ]:
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('CHROMA_OPENAI_API_KEY'),
    api_base=os.getenv('OPENAI_API_BASE')
)

In [ ]:
# Get or create collection (avoids error if collection already exists)
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn)

    
print(f"Collection '{collection.name}' is ready — {collection.count()} documents currently indexed.")

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [ ]:
from pydantic import BaseModel, Field

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    description: str = Field(description="Explanation of why the documents are or are not sufficient")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    analyze the usability of the documents to respond to that question.

    Args:
        question: original question from user
        retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    Returns:
        A dict with:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    docs_text = "\n\n".join(retrieved_docs)

    prompt = f"""
    Your task is to evaluate whether the retrieved documents are enough to answer the user's question.

    User question:
    {question}

    Retrieved documents:
    {docs_text}

    Decide:
    1. Are these documents useful enough to answer the question?
    2. Explain why or why not.
    3. Mention if important information is missing, incomplete, or irrelevant.
    """

    result = llm.responses.parse(
        model="gpt-4.1-mini",
        input=prompt,
        text_format=EvaluationReport,
    )

    report = result.output_parsed

    return {
        "useful": report.useful,
        "description": report.description,
    }


#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

In [ ]:
@tool
def game_web_search(question: str, max_results: int = 3) -> dict:
    """Search the web using Tavily and return a summarized response."""

    client = TavilyClient(api_key=TAVILY_API_KEY)
    search_result = client.search(
        query=question,
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    return {
        "answer": search_result.get("answer"),
        "results": search_result.get("results", [])[:max_results],
        "search_metadata": {
            "query": question,
            "timestamp": __import__("datetime").datetime.now().isoformat(),
        },
    }


### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [15]:
import json
from enum import Enum, auto
from typing import List, Dict, Any

# --- 1. STATE DEFINITIONS ---
class AgentState(Enum):
    PLANNING = auto()
    EXECUTING_TOOL = auto()
    FINALIZING = auto()
    COMPLETE = auto()



In [16]:
# --- 2. AGENT ABSTRACTION CLASS ---
class Agent:
    def __init__(self, model: Any, tools: List[Any], instructions: str, max_steps: int = 10):
        self.model = model
        self.tools = {tool.__name__: tool for tool in tools}
        self.instructions = instructions
        self.max_steps = max_steps
        
        # Internal state initialization
        self.memory: List[Dict[str, str]] = []
        self.current_state = AgentState.PLANNING
        self.step_count = 0

    def run(self, user_prompt: str) -> str:
        """Main execution loop driven by state machine transitions."""
        # Fresh history per run, anchored by system instructions
        self.memory = [{"role": "system", "content": self.instructions}]
        self.memory.append({"role": "user", "content": user_prompt})
        
        self.current_state = AgentState.PLANNING
        self.step_count = 0
        final_answer = ""

        print(f"\n🚀 [STARTING AGENT] Query: '{user_prompt}'")

        while self.current_state != AgentState.COMPLETE:
            if self.step_count >= self.max_steps:
                print("❌ [GUARDRAIL] Hit maximum step execution limit.")
                return "Error: Agent execution exceeded max allowed steps."
            
            self.step_count += 1
            print(f" -> [STEP {self.step_count}] Current State: {self.current_state.name}")

            if self.current_state == AgentState.PLANNING:
                final_answer = self._handle_planning()
            elif self.current_state == AgentState.EXECUTING_TOOL:
                self._handle_execution()
            elif self.current_state == AgentState.FINALIZING:
                final_answer = self._handle_finalizing()

        return final_answer

    def _handle_planning(self) -> str:
        """Asks the model to decide whether to invoke a tool or answer directly."""
        response = self.model.generate(messages=self.memory, tools=list(self.tools.keys()))
        
        assistant_msg = {"role": "assistant", "content": response.get("content")}
        
        if "tool_calls" in response:
            assistant_msg["tool_calls"] = response["tool_calls"]
            self.current_state = AgentState.EXECUTING_TOOL
            print(f"    [PLAN] Model requested tool: {response['tool_calls'][0]['name']}")
        else:
            self.current_state = AgentState.FINALIZING
            print("    [PLAN] Model ready to finalize answer.")
            
        self.memory.append(assistant_msg)
        return response.get("content", "")

    def _handle_execution(self):
        """Executes tool requests and feeds results back to agent memory."""
        last_msg = self.memory[-1]
        
        for tool_call in last_msg.get("tool_calls", []):
            tool_name = tool_call["name"]
            tool_args = tool_call["arguments"]
            
            if tool_name in self.tools:
                try:
                    # Dynamically execute matching tool
                    result = self.tools[tool_name](**tool_args)
                    print(f"    [EXECUTE] Tool '{tool_name}' completed successfully.")
                except Exception as e:
                    result = f"Error running tool: {str(e)}"
            else:
                result = f"Error: Tool '{tool_name}' not found."
                
            self.memory.append({
                "role": "tool", 
                "tool_call_id": tool_call["id"], 
                "content": str(result)
            })
            
        # Re-enter planning state with fresh tool data
        self.current_state = AgentState.PLANNING

    def _handle_finalizing(self) -> str:
        """Concludes state loop and marks execution complete."""
        self.current_state = AgentState.COMPLETE
        print("🏁 [COMPLETE] Terminating loop sequence.")
        return self.memory[-1]["content"]



In [17]:

# --- 3. GAME TOOLS SUITE ---
def search_gaming_database(query: str) -> str:
    """Mock database tool populated with target game facts."""
    data = {
        "pokemon gold and silver": "Released in Japan: Nov 21, 1999. North America: Oct 15, 2000. Europe: Apr 6, 2001.",
        "mario 3d platformer": "Super Mario 64 was the first 3D platformer Mario game, released June 23, 1996.",
        "mortal kombat x ps5": "Released in 2015 for PS4, Xbox One, PC. No native PS5 version, but backward compatible."
    }
    for key, output in data.items():
        if key in query.lower() or query.lower() in key:
            return output
    return "No database records found matching that query."




In [18]:
# --- 4. MODEL IMPLEMENTATION MOCK ---
class MockLLM:
    """Simulates realistic assistant messages and tool call sequences."""
    def __init__(self):
        self.call_count = 0

    def generate(self, messages: List[Dict], tools: List[str]) -> Dict:
        self.call_count += 1
        user_msg = messages[-1]["content"].lower()

        # Phase 1: Direct request received -> trigger tool search
        if messages[-1]["role"] == "user":
            if "gold" in user_msg and "silver" in user_msg:
                query_term = "pokemon gold and silver"
            elif "mario" in user_msg:
                query_term = "mario 3d platformer"
            else:
                query_term = "mortal kombat x ps5"

            return {
                "content": "Looking up details in historical records.",
                "tool_calls": [{
                    "id": f"call_{self.call_count}",
                    "name": "search_gaming_database",
                    "arguments": {"query": query_term}
                }]
            }
        
        # Phase 2: Tool results populated -> format final answer string
        if messages[-1]["role"] == "tool":
            tool_output = messages[-1]["content"]
            return {
                "content": f"Verified Fact: {tool_output}"
            }
        
        return {"content": "Task completed."}


In [22]:


# --- 5. EXECUTION BLOCK ---
if __name__ == "__main__":
    # Settings Setup
    instructions_prompt = (
        "You are a gaming history expert. Always use tools to verify release dates."
    )
    developed_tools = [search_gaming_database]
    equipped_model = MockLLM()

    # Initializing Agent Abstraction
    agent = Agent(
        model=equipped_model, 
        tools=developed_tools, 
        instructions=instructions_prompt
    )

    # Invocations
    queries = [
        "When Pokémon Gold and Silver was released?",
        
    ]
for q in queries:
        result = agent.run(q)
        print(f"🤖 [AGENT FINAL ANSWER]: {result}")
        print("=" * 65)

    


🚀 [STARTING AGENT] Query: 'When Pokémon Gold and Silver was released?'
 -> [STEP 1] Current State: PLANNING
    [PLAN] Model requested tool: search_gaming_database
 -> [STEP 2] Current State: EXECUTING_TOOL
    [EXECUTE] Tool 'search_gaming_database' completed successfully.
 -> [STEP 3] Current State: PLANNING
    [PLAN] Model ready to finalize answer.
 -> [STEP 4] Current State: FINALIZING
🏁 [COMPLETE] Terminating loop sequence.
🤖 [AGENT FINAL ANSWER]: Verified Fact: Released in Japan: Nov 21, 1999. North America: Oct 15, 2000. Europe: Apr 6, 2001.


In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

In [25]:
questions = [
    "When Pokémon Gold and Silver was released?",
    
]

for question in questions:
    result = agent.run(question)
    print(f"Question: {question}")
    print(f"Answer: {result}")
    print("-" * 50)



🚀 [STARTING AGENT] Query: 'When Pokémon Gold and Silver was released?'
 -> [STEP 1] Current State: PLANNING
    [PLAN] Model requested tool: search_gaming_database
 -> [STEP 2] Current State: EXECUTING_TOOL
    [EXECUTE] Tool 'search_gaming_database' completed successfully.
 -> [STEP 3] Current State: PLANNING
    [PLAN] Model ready to finalize answer.
 -> [STEP 4] Current State: FINALIZING
🏁 [COMPLETE] Terminating loop sequence.
Question: When Pokémon Gold and Silver was released?
Answer: Verified Fact: Released in Japan: Nov 21, 1999. North America: Oct 15, 2000. Europe: Apr 6, 2001.
--------------------------------------------------


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes